← [Overview](00_overview.ipynb)

# Preprocessing: normalization and unstacking

Before any clustering can happen, tsam runs two shared preprocessing steps: it **normalizes** every attribute to a common scale, then **unstacks** the
flat series so each period becomes one row-vector. Every clustering method in this series
operates on the matrix produced here.

First, the tiny six-day dataset this whole series uses — six days × 4 timesteps/day
(6-hourly), two attributes (a **solar** proxy and a **load** proxy). The day shapes are
deliberately distinct: days 0–1 sunny, days 2–3 overcast, days 4–5 cloudy with one
extreme-load day. Small enough to verify every number by hand.

This notebook is the **single source of truth** for the data the rest of the series uses.
It saves two files that the sibling notebooks load instead of recomputing anything:

* `../tiny.csv` — the raw six-day series (for `tsam.aggregate` calls and plots).
* `../tiny_periods.csv` — the **preprocessed period matrix** $D$ (normalized + unstacked),
  which the clustering notebooks feed straight into the distance objective.

## 1 Build and save the tiny time series

In [1]:
import os
import tempfile

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

# Tiny 6-day × 4-timestep synthetic dataset (the running example for the series).
idx = pd.date_range("2020-01-01", periods=24, freq="6h")
solar_values = [0, 8, 6, 0, 0, 7, 7, 0, 0, 3, 2, 0, 0, 2, 3, 0, 0, 1, 1, 0, 0, 1, 0, 0]
load_values = [3, 3, 4, 5, 3, 3, 4, 4, 4, 5, 5, 6, 4, 4, 6, 5, 6, 6, 7, 7, 6, 7, 8, 10]
tiny = pd.DataFrame({"solar": solar_values, "load": load_values}, index=idx)
tiny.index.name = "time"

# Persist it next to testdata.csv so notebooks 02-07 just *read* this file instead
# of redefining the arrays. The docs build executes notebooks in parallel, so a
# sibling may read tiny.csv while this cell runs — write atomically (temp file +
# os.replace) so a reader never sees a half-written file.
fd, _tmp = tempfile.mkstemp(dir="..", suffix=".csv")
os.close(fd)
tiny.to_csv(_tmp)
os.replace(_tmp, "../tiny.csv")
print("Saved ../tiny.csv:", tiny.shape, "(6 days × 4 timesteps, 2 attributes)")

# One row per timestep; all 48 values at a glance.
print(tiny)

Saved ../tiny.csv: (24, 2) (6 days × 4 timesteps, 2 attributes)
                     solar  load
time                            
2020-01-01 00:00:00      0     3
2020-01-01 06:00:00      8     3
2020-01-01 12:00:00      6     4
2020-01-01 18:00:00      0     5
2020-01-02 00:00:00      0     3
2020-01-02 06:00:00      7     3
2020-01-02 12:00:00      7     4
2020-01-02 18:00:00      0     4
2020-01-03 00:00:00      0     4
2020-01-03 06:00:00      3     5
2020-01-03 12:00:00      2     5
2020-01-03 18:00:00      0     6
2020-01-04 00:00:00      0     4
2020-01-04 06:00:00      2     4
2020-01-04 12:00:00      3     6
2020-01-04 18:00:00      0     5
2020-01-05 00:00:00      0     6
2020-01-05 06:00:00      1     6
2020-01-05 12:00:00      1     7
2020-01-05 18:00:00      0     7
2020-01-06 00:00:00      0     6
2020-01-06 06:00:00      1     7
2020-01-06 12:00:00      0     8
2020-01-06 18:00:00      0    10


In [2]:
fig = px.line(
    tiny.reset_index(names="time"),
    x="time",
    y=["solar", "load"],
    facet_col="variable",
    title="The tiny synthetic dataset (6 days × 4 timesteps)",
    labels={"value": "value", "time": "time"},
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

## 2  Attribute-wise min-max normalization

Each attribute is scaled to $[0, 1]$ so that no column dominates the distance
computation:

$$
x_{a,s} = \frac{x'_{a,s} - \min x'_a}{\max x'_a - \min x'_a}
$$

where $x'_{a,s}$ is the raw value of attribute $a$ at time step $s$, and $\min x'_a$,
$\max x'_a$ are the column min/max over all time steps.

In [3]:
col_min = tiny.min()
col_max = tiny.max()

print("Column minimums:")
print(col_min.to_string())
print("\nColumn maximums:")
print(col_max.to_string())

normalized = (tiny - col_min) / (col_max - col_min)
print("\nNormalized dataset (all values in [0, 1]):")
normalized.round(4)

Column minimums:
solar    0
load     3

Column maximums:
solar     8
load     10

Normalized dataset (all values in [0, 1]):


,solar,load
time,,
2020-01-01 00:00:00,0.000,0.0000
2020-01-01 06:00:00,1.000,0.0000
2020-01-01 12:00:00,0.750,0.1429
2020-01-01 18:00:00,0.000,0.2857
2020-01-02 00:00:00,0.000,0.0000
2020-01-02 06:00:00,0.875,0.0000
2020-01-02 12:00:00,0.875,0.1429
2020-01-02 18:00:00,0.000,0.1429
2020-01-03 00:00:00,0.000,0.1429


## 3  Unstacking to period row-vectors (the D matrix)

The flat normalized series is **reshaped** so each period (day) becomes one row vector
whose dimensions are:

$$
\text{dim} = N_t \times N_a \quad (\text{timesteps per period} \times \text{attributes})
$$

With 4 timesteps/day and 2 attributes, each period is an 8-dimensional point. Clustering
groups these six points in that 8-dimensional space.

This matrix is the hand-off point: it is saved to `../tiny_periods.csv`, and the
clustering notebooks (starting with [Partitional clustering](02_partitional_clustering.ipynb))
load it directly — they never re-run the normalization or unstacking above.

In [4]:
# tsam's unstack_to_periods does the same reshape
unstacked = tsam.unstack_to_periods(data=normalized, period_duration="1D")

# Persist the preprocessed period matrix so the clustering notebooks load it
# directly instead of re-running normalization + unstacking. Written atomically
# for the same parallel-build reason as tiny.csv above.
fd, _tmp = tempfile.mkstemp(dir="..", suffix=".csv")
os.close(fd)
unstacked.to_csv(_tmp)
os.replace(_tmp, "../tiny_periods.csv")
print("Saved ../tiny_periods.csv — period matrix shape:", unstacked.shape,
      "(rows=days, cols=(attribute, timestep) pairs)")
unstacked.round(4)

Saved ../tiny_periods.csv — period matrix shape: (6, 8) (rows=days, cols=(attribute, timestep) pairs)


solar                       load                        
TimeStep      0      1      2    3       0       1       2       3
PeriodNum                                                         
0           0.0  1.000  0.750  0.0  0.0000  0.0000  0.1429  0.2857
1           0.0  0.875  0.875  0.0  0.0000  0.0000  0.1429  0.1429
2           0.0  0.375  0.250  0.0  0.1429  0.2857  0.2857  0.4286
3           0.0  0.250  0.375  0.0  0.1429  0.1429  0.4286  0.2857
4           0.0  0.125  0.125  0.0  0.4286  0.4286  0.5714  0.5714
5           0.0  0.125  0.000  0.0  0.4286  0.5714  0.7143  1.0000

---

**Next:** [Partitional clustering](02_partitional_clustering.ipynb) — k-means, k-medoids and k-maxoids on this matrix.

### Further reading

* [Mathematical Background](../../background/math.md) — full notation and formulas
* [Pipeline Guide](../../background/architecture/pipeline_guide.md) — the four pipeline phases